# Stage 1 — DeepTopic model summary with improved cell-type scoring

This version is for your actual `topic × region` AnnData object.

Main change:

Instead of only using simple `mean(all topics in a cell type)`, this notebook computes several cell-type scores:

1. `mean_all`: average all topics assigned to a cell type  
2. `mean_usable`: average Stage 0 usable topics only; if a cell type has no usable topic, fallback to all topics  
3. `max_topic`: the strongest topic score within each cell type  
4. `top2_mean`: average the top 2 topic scores within each cell type  
5. `weighted_stage0`: weighted average using Stage 0 specificity ratio  

This keeps small but specific topics visible instead of letting only large dominant topics control the cell-type projection.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, accuracy_score, top_k_accuracy_score

os.environ["KERAS_BACKEND"] = "torch"

import keras
import crested

## 1. Configuration

In [2]:
SPECIES = "human"   # "human" or "macaque"

BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")
OUTROOT = BASE / "runs" / "out" / "stage1_model_summary"

CONFIG = {
    "human": {
        "adata_path": BASE / "runs" / "out" / "human_topics.h5ad",
        "model_path": BASE / "runs" / "out" / "deeptopic_human" / "final_model.keras",
        "topic_annotation_path": BASE / "data" / "stage0_annotation" / "topic_annotation_human.tsv",
        "genome_fa": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa"),
        "batch_size": 4,
    },
    "macaque": {
        "adata_path": BASE / "runs" / "out" / "macaque_topics.h5ad",
        "model_path": BASE / "runs" / "out" / "deeptopic_macaque" / "final_model.keras",
        "topic_annotation_path": BASE / "data" / "stage0_annotation" / "topic_annotation_macaque.tsv",
        "genome_fa": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.fa"),
        "batch_size": 4,
    },
}

cfg = CONFIG[SPECIES]
OUTDIR = OUTROOT / SPECIES
OUTDIR.mkdir(parents=True, exist_ok=True)

print("SPECIES:", SPECIES)
print("OUTDIR:", OUTDIR)
print("adata:", cfg["adata_path"])
print("model:", cfg["model_path"])
print("topic annotation:", cfg["topic_annotation_path"])
print("genome:", cfg["genome_fa"])
print("batch_size:", cfg["batch_size"])

SPECIES: human
OUTDIR: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human
adata: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/human_topics.h5ad
model: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/deeptopic_human/final_model.keras
topic annotation: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/stage0_annotation/topic_annotation_human.tsv
genome: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa
batch_size: 4


## 2. Load data

In [3]:
adata = ad.read_h5ad(cfg["adata_path"])
topic_anno = pd.read_csv(cfg["topic_annotation_path"], sep="\t")

model = keras.models.load_model(cfg["model_path"], compile=False)
genome = crested.Genome(str(cfg["genome_fa"]))

topic_anno["topic"] = topic_anno["topic"].astype(str)
topic_anno["top1_celltype"] = topic_anno["top1_celltype"].astype(str)

if "usable_for_motif" not in topic_anno.columns:
    topic_anno["usable_for_motif"] = False

if "specificity_ratio_top1_over_top2" not in topic_anno.columns:
    topic_anno["specificity_ratio_top1_over_top2"] = 1.0

topic_anno["specificity_ratio_top1_over_top2"] = pd.to_numeric(
    topic_anno["specificity_ratio_top1_over_top2"],
    errors="coerce"
).fillna(1.0)

print(adata)
print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))
print("obs index head:", adata.obs.index[:5].tolist())
print("var index head:", adata.var.index[:5].tolist())
display(topic_anno.head())

AnnData object with n_obs × n_vars = 100 × 415405
    obs: 'file_path', 'n_open_regions'
    var: 'n_classes', 'chr', 'start', 'end', 'split'
obs columns: ['file_path', 'n_open_regions']
var columns: ['n_classes', 'chr', 'start', 'end', 'split']
obs index head: ['Topic1', 'Topic10', 'Topic100', 'Topic11', 'Topic12']
var index head: ['chr1:9973-10473', 'chr1:180766-181266', 'chr1:191161-191661', 'chr1:628999-629499', 'chr1:629674-630174']


,topic,topic_num,species,top1_celltype,top1_score,top2_celltype,top2_score,specificity_ratio_top1_over_top2,score_delta_top1_minus_top2,annotation_confidence,annotation_class,lineage_group,lineage_group_top2,bed_path,bed_exists,n_peaks,usable_for_motif
0,Topic1,1,human,OPC,0.018942,Vascular,0.016960,1.116855,0.001982,low,ambiguous,glia,vascular,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/...,True,8150,False
1,Topic2,2,human,CGE_interneuron,0.011338,LGE_FOXP2_TSHZ1_MSN,0.010950,1.035347,0.000387,low,ambiguous,interneuron,subpallial_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/...,True,6984,False
2,Topic3,3,human,OPC,0.015154,ExNeu_IT,0.012243,1.237767,0.002911,medium,ambiguous,glia,excitatory_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/...,True,15275,False
3,Topic4,4,human,ExIPC,0.017248,MGE_progenitors,0.013981,1.233697,0.003267,medium,ambiguous,progenitor,progenitor,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/...,True,4338,False
4,Topic5,5,human,ExNeu_IT,0.027088,ExNeu_Non_IT,0.015738,1.721155,0.011350,high,clean_top1,excitatory_neuron,excitatory_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/...,True,10672,True


## 3. Helper functions

In [4]:
def save_df(df: pd.DataFrame, path: Path):
    df.to_csv(path, sep="\t", index=False)
    print("saved:", path)


def get_test_subset(adata):
    if "split" not in adata.var.columns:
        raise ValueError("adata.var['split'] not found")
    mask = adata.var["split"].astype(str).str.lower() == "test"
    return adata[:, mask].copy(), np.asarray(mask)


def infer_true_topic_from_topic_region_matrix(adata_sub):
    X = adata_sub.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    topic_idx = X.argmax(axis=0)
    topic_names = adata_sub.obs.index.astype(str).tolist()
    true_topic = pd.Series(
        [topic_names[i] for i in topic_idx],
        index=adata_sub.var.index,
        name="true_topic"
    )
    return true_topic


def predict_topic_probabilities(adata_sub, model, genome, batch_size=4):
    pred = crested.tl.predict(
        adata_sub,
        model,
        genome=genome,
        batch_size=batch_size
    )
    if isinstance(pred, np.ndarray):
        pred_df = pd.DataFrame(
            pred,
            index=adata_sub.var.index,
            columns=adata_sub.obs.index.astype(str).tolist()
        )
    elif isinstance(pred, pd.DataFrame):
        pred_df = pred.copy()
        pred_df.index = adata_sub.var.index
    else:
        pred_df = pd.DataFrame(
            pred,
            index=adata_sub.var.index,
            columns=adata_sub.obs.index.astype(str).tolist()
        )
    return pred_df


def add_prediction_rank_columns(pred_df):
    out = pred_df.copy()
    vals = out.to_numpy()
    order = np.argsort(-vals, axis=1)
    out["top_topic_pred"] = out.columns[order[:, 0]]
    out["top_score_pred"] = vals[np.arange(vals.shape[0]), order[:, 0]]
    out["second_score_pred"] = vals[np.arange(vals.shape[0]), order[:, 1]]
    out["margin_top1_top2_pred"] = out["top_score_pred"] - out["second_score_pred"]
    return out


def make_confusion_pairs(cm_df):
    rows = []
    for true_lab in cm_df.index:
        row = cm_df.loc[true_lab].copy()
        if true_lab in row.index:
            row.loc[true_lab] = 0
        pred_lab = row.idxmax()
        rows.append({
            "true_topic": true_lab,
            "most_confused_with": pred_lab,
            "count": int(row.max()),
        })
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)


def compute_celltype_scores(eval_df, topic_anno, class_names, strategy="mean_all"):
    # strategy options:
    # mean_all        = mean of all topics assigned to celltype
    # mean_usable     = mean of usable_for_motif topics only; fallback to all if none
    # max_topic       = max topic score within celltype
    # top2_mean       = mean of top 2 topic scores within celltype
    # weighted_stage0 = weighted mean using specificity_ratio_top1_over_top2
    celltypes = sorted(topic_anno["top1_celltype"].dropna().astype(str).unique().tolist())
    out = pd.DataFrame(index=eval_df.index)

    for celltype in celltypes:
        sub_anno_all = topic_anno.loc[topic_anno["top1_celltype"] == celltype].copy()
        sub_anno = sub_anno_all.copy()

        if strategy == "mean_usable":
            usable = sub_anno_all.loc[sub_anno_all["usable_for_motif"] == True].copy()
            if usable.shape[0] > 0:
                sub_anno = usable

        topics = [t for t in sub_anno["topic"].astype(str).tolist() if t in class_names]
        if len(topics) == 0:
            continue

        vals = eval_df[topics].to_numpy()

        if strategy in ["mean_all", "mean_usable"]:
            score = vals.mean(axis=1)

        elif strategy == "max_topic":
            score = vals.max(axis=1)

        elif strategy == "top2_mean":
            if vals.shape[1] == 1:
                score = vals[:, 0]
            else:
                k = min(2, vals.shape[1])
                score = np.sort(vals, axis=1)[:, -k:].mean(axis=1)

        elif strategy == "weighted_stage0":
            weights = (
                sub_anno.set_index("topic")
                .loc[topics, "specificity_ratio_top1_over_top2"]
                .astype(float)
                .to_numpy()
            )
            weights = np.nan_to_num(weights, nan=1.0)
            weights = np.clip(weights, 1.0, np.nanpercentile(weights, 95))
            score = np.average(vals, axis=1, weights=weights)

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        out[f"cellscore__{celltype}"] = score

    return out


def summarize_celltype_scores(cellscore_df, topic_anno, species, strategy):
    cellscore_cols = [c for c in cellscore_df.columns if c.startswith("cellscore__")]
    if len(cellscore_cols) == 0:
        return pd.DataFrame(), cellscore_df

    vals = cellscore_df[cellscore_cols].to_numpy()
    order = np.argsort(-vals, axis=1)

    scored = cellscore_df.copy()
    scored["best_celltype"] = [cellscore_cols[i].replace("cellscore__", "") for i in order[:, 0]]
    scored["best_celltype_score"] = vals[np.arange(vals.shape[0]), order[:, 0]]

    if len(cellscore_cols) > 1:
        scored["second_best_celltype"] = [cellscore_cols[i].replace("cellscore__", "") for i in order[:, 1]]
        scored["second_best_celltype_score"] = vals[np.arange(vals.shape[0]), order[:, 1]]
        scored["celltype_margin_top1_top2"] = (
            scored["best_celltype_score"] - scored["second_best_celltype_score"]
        )
    else:
        scored["second_best_celltype"] = np.nan
        scored["second_best_celltype_score"] = np.nan
        scored["celltype_margin_top1_top2"] = np.nan

    rows = []
    for col in cellscore_cols:
        celltype = col.replace("cellscore__", "")
        mask = scored["best_celltype"] == celltype
        rows.append({
            "species": species,
            "strategy": strategy,
            "celltype": celltype,
            "n_topics_assigned_in_stage0": int((topic_anno["top1_celltype"] == celltype).sum()),
            "n_usable_topics": int(((topic_anno["top1_celltype"] == celltype) & (topic_anno["usable_for_motif"] == True)).sum()),
            "mean_projected_score": float(scored[col].mean()),
            "median_projected_score": float(scored[col].median()),
            "n_regions_best": int(mask.sum()),
            "mean_best_margin_when_winning": float(scored.loc[mask, "celltype_margin_top1_top2"].mean()) if mask.any() else np.nan,
        })

    summary = pd.DataFrame(rows).sort_values(
        ["n_regions_best", "mean_projected_score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return summary, scored

## 4. Build test set and infer pseudo-labels

In [5]:
adata_test, full_test_mask = get_test_subset(adata)

print("full adata:", adata.shape)
print("test adata:", adata_test.shape)

true_topic = infer_true_topic_from_topic_region_matrix(adata_test)
true_topic_counts = true_topic.value_counts().rename_axis("true_topic").reset_index(name="n_regions")
display(true_topic_counts.head(20))

full adata: (100, 415405)
test adata: (100, 38752)


,true_topic,n_regions
0,Topic7,6188
1,Topic90,5322
2,Topic65,5122
3,Topic31,2378
4,Topic12,1446
5,Topic14,1377
6,Topic17,1275
7,Topic100,976
8,Topic11,941
9,Topic38,888


## 5. Predict test regions

In [6]:
pred_df = predict_topic_probabilities(
    adata_test,
    model=model,
    genome=genome,
    batch_size=cfg["batch_size"]
)

print(type(pred_df))
print(pred_df.shape)
display(pred_df.iloc[:5, :5])

2026-04-27T13:15:21.029443+0200 INFO Lazily importing module crested.tl. This could take a second...
9688/9688 ━━━━━━━━━━━━━━━━━━━━ 151s 16ms/step
<class 'pandas.core.frame.DataFrame'>
(38752, 100)


,Topic1,Topic10,Topic100,Topic11,Topic12
region,,,,,
chr18:9980-10480,0.032774,0.046902,0.061915,0.130943,0.121030
chr18:111530-112030,0.015426,0.026453,0.030137,0.163484,0.137264
chr18:158522-159022,0.019196,0.022925,0.027121,0.250932,0.158931
chr18:172111-172611,0.025811,0.006442,0.009630,0.003894,0.019794
chr18:190063-190563,0.004056,0.021073,0.050289,0.001317,0.018924


## 6. Assemble region-level prediction table

In [7]:
class_names = adata_test.obs.index.astype(str).tolist()

eval_df = add_prediction_rank_columns(pred_df)
eval_df["true_topic"] = true_topic.values
eval_df["region"] = eval_df.index.astype(str)

topic_prob = eval_df[class_names].copy()
y_true = eval_df["true_topic"].astype(str).values
y_pred = eval_df["top_topic_pred"].astype(str).values

display(eval_df.head())

,Topic1,Topic10,Topic100,Topic11,Topic12,Topic13,Topic14,Topic15,Topic16,Topic17,...,Topic96,Topic97,Topic98,Topic99,top_topic_pred,top_score_pred,second_score_pred,margin_top1_top2_pred,true_topic,region
region,,,,,,,,,,,,,,,,,,,,,
chr18:9980-10480,0.032774,0.046902,0.061915,0.130943,0.121030,0.054525,0.083403,0.038791,0.091347,0.174996,...,0.150809,0.164242,0.038628,0.026259,Topic90,0.298888,0.293340,0.005548,Topic40,chr18:9980-10480
chr18:111530-112030,0.015426,0.026453,0.030137,0.163484,0.137264,0.028921,0.064804,0.023765,0.090811,0.184265,...,0.174734,0.178940,0.021912,0.014800,Topic7,0.286315,0.265868,0.020448,Topic16,chr18:111530-112030
chr18:158522-159022,0.019196,0.022925,0.027121,0.250932,0.158931,0.026331,0.062065,0.021370,0.106541,0.220995,...,0.235909,0.219769,0.020293,0.013778,Topic35,0.298352,0.283096,0.015255,Topic25,chr18:158522-159022
chr18:172111-172611,0.025811,0.006442,0.009630,0.003894,0.019794,0.008912,0.015829,0.005231,0.012525,0.035193,...,0.007283,0.022241,0.006786,0.004272,Topic90,0.371740,0.269003,0.102737,Topic90,chr18:172111-172611
chr18:190063-190563,0.004056,0.021073,0.050289,0.001317,0.018924,0.044788,0.039210,0.043226,0.005241,0.017410,...,0.003979,0.013475,0.073909,0.030645,Topic90,0.508054,0.275979,0.232075,Topic90,chr18:190063-190563


## 7. Overall metrics

In [8]:
class_to_idx = {c: i for i, c in enumerate(class_names)}

overall_top1_acc = accuracy_score(y_true, y_pred)
y_true_idx = np.array([class_to_idx[x] for x in y_true])
topic_prob_np = topic_prob.to_numpy()

overall_top3_acc = top_k_accuracy_score(y_true_idx, topic_prob_np, k=3, labels=np.arange(len(class_names)))
overall_top5_acc = top_k_accuracy_score(y_true_idx, topic_prob_np, k=5, labels=np.arange(len(class_names)))

overall_metrics = pd.DataFrame([
    {"species": SPECIES, "metric": "top1_accuracy", "value": overall_top1_acc},
    {"species": SPECIES, "metric": "top3_accuracy", "value": overall_top3_acc},
    {"species": SPECIES, "metric": "top5_accuracy", "value": overall_top5_acc},
    {"species": SPECIES, "metric": "n_test_regions", "value": eval_df.shape[0]},
])

display(overall_metrics)
save_df(overall_metrics, OUTDIR / "overall_metrics.tsv")
save_df(eval_df.reset_index(drop=True), OUTDIR / "region_level_predictions.tsv")

,species,metric,value
0,human,top1_accuracy,0.171552
1,human,top3_accuracy,0.471382
2,human,top5_accuracy,0.567145
3,human,n_test_regions,38752.000000


saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/overall_metrics.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/region_level_predictions.tsv


## 8. Topic-level summaries

In [9]:
true_topic_counts = (
    eval_df["true_topic"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="n_true_regions")
)

pred_topic_counts = (
    eval_df["top_topic_pred"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="n_pred_regions")
)

true_vs_pred_topic_counts = true_topic_counts.merge(pred_topic_counts, on="topic", how="outer").fillna(0)
true_vs_pred_topic_counts["n_true_regions"] = true_vs_pred_topic_counts["n_true_regions"].astype(int)
true_vs_pred_topic_counts["n_pred_regions"] = true_vs_pred_topic_counts["n_pred_regions"].astype(int)
true_vs_pred_topic_counts = true_vs_pred_topic_counts.merge(
    topic_anno[["topic", "top1_celltype", "lineage_group", "annotation_confidence", "usable_for_motif"]],
    how="left",
    on="topic",
)
true_vs_pred_topic_counts = true_vs_pred_topic_counts.sort_values(
    ["n_pred_regions", "n_true_regions"],
    ascending=[False, False]
).reset_index(drop=True)

save_df(true_vs_pred_topic_counts, OUTDIR / "true_vs_pred_topic_counts.tsv")
display(true_vs_pred_topic_counts.head(20))

saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/true_vs_pred_topic_counts.tsv


,topic,n_true_regions,n_pred_regions,top1_celltype,lineage_group,annotation_confidence,usable_for_motif
0,Topic90,5322,28555,LGE_FOXP2_TSHZ1_MSN,subpallial_neuron,low,False
1,Topic7,6188,4318,LGE_OB_interneuron,interneuron,low,False
2,Topic65,5122,3812,MGE_interneuron,interneuron,high,True
3,Topic40,53,609,OPC,glia,low,False
4,Topic92,150,411,Microglia,immune,low,False
5,Topic35,41,397,LGE_OB_interneuron,interneuron,low,False
6,Topic17,1275,384,OPC,glia,low,False
7,Topic78,237,77,Vascular,vascular,low,False
8,Topic83,117,50,LGE_OB_interneuron,interneuron,low,False
9,Topic11,941,46,OPC,glia,low,False


In [10]:
per_topic_rows = []
for topic in class_names:
    sub = eval_df.loc[eval_df["true_topic"] == topic].copy()
    if sub.empty:
        continue

    top1_acc = (sub["top_topic_pred"] == topic).mean()

    vals = sub[class_names].to_numpy()
    order = np.argsort(-vals, axis=1)[:, :3]
    topic_idx = class_to_idx[topic]
    top3_acc = np.mean([topic_idx in row for row in order])

    per_topic_rows.append({
        "species": SPECIES,
        "topic": topic,
        "n_regions": sub.shape[0],
        "top1_accuracy": top1_acc,
        "top3_accuracy": top3_acc,
        "mean_true_topic_score": sub[topic].mean(),
        "median_true_topic_score": sub[topic].median(),
        "mean_pred_margin_top1_top2": sub["margin_top1_top2_pred"].mean(),
    })

per_topic = pd.DataFrame(per_topic_rows)

per_topic = per_topic.merge(
    topic_anno[["topic", "top1_celltype", "lineage_group", "annotation_confidence", "usable_for_motif"]],
    how="left",
    on="topic",
)

def stage2_priority(row):
    if bool(row["usable_for_motif"]) and row["top3_accuracy"] >= 0.50:
        return "high"
    if bool(row["usable_for_motif"]) and row["top3_accuracy"] >= 0.10:
        return "medium"
    if bool(row["usable_for_motif"]):
        return "candidate_low_model_recall"
    return "not_priority"

per_topic["stage2_priority"] = per_topic.apply(stage2_priority, axis=1)

per_topic = per_topic.sort_values(
    ["stage2_priority", "usable_for_motif", "top3_accuracy", "n_regions"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

save_df(per_topic, OUTDIR / "per_topic_summary.tsv")
save_df(per_topic[per_topic["usable_for_motif"] == True].copy(), OUTDIR / "stage2_topic_candidates.tsv")
display(per_topic.head(30))

saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/per_topic_summary.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/stage2_topic_candidates.tsv


,species,topic,n_regions,top1_accuracy,top3_accuracy,mean_true_topic_score,median_true_topic_score,mean_pred_margin_top1_top2,top1_celltype,lineage_group,annotation_confidence,usable_for_motif,stage2_priority
0,human,Topic80,263,0.000000,0.076046,0.075638,0.063478,0.187237,CGE_LGE_progenitors,progenitor,high,True,candidate_low_model_recall
1,human,Topic9,146,0.000000,0.027397,0.073415,0.061016,0.202320,ExIPC,progenitor,high,True,candidate_low_model_recall
2,human,Topic5,135,0.000000,0.022222,0.057701,0.052871,0.120620,ExNeu_IT,excitatory_neuron,high,True,candidate_low_model_recall
3,human,Topic60,92,0.000000,0.010870,0.055252,0.043518,0.041840,dorsal_RG,progenitor,high,True,candidate_low_model_recall
4,human,Topic95,117,0.000000,0.008547,0.086701,0.085106,0.098524,TriIPC_Astrocyte,glia_like_progenitor,high,True,candidate_low_model_recall
5,human,Topic55,220,0.000000,0.004545,0.044400,0.035984,0.152229,ExIPC,progenitor,high,True,candidate_low_model_recall
6,human,Topic38,888,0.000000,0.000000,0.063512,0.062299,0.092596,Vascular,vascular,high,True,candidate_low_model_recall
7,human,Topic30,611,0.000000,0.000000,0.047124,0.041653,0.130710,Neuroblast,neuronal_intermediate,high,True,candidate_low_model_recall
8,human,Topic16,569,0.000000,0.000000,0.051076,0.041550,0.051152,ExNeu_IT,excitatory_neuron,high,True,candidate_low_model_recall
9,human,Topic23,530,0.000000,0.000000,0.050475,0.044257,0.190610,ExNeu_IT,excitatory_neuron,high,True,candidate_low_model_recall


## 9. Confusion matrix

In [11]:
labels_present = class_names
cm = confusion_matrix(y_true, y_pred, labels=labels_present)
cm_df = pd.DataFrame(cm, index=labels_present, columns=labels_present)

cm_out = cm_df.reset_index().rename(columns={"index": "true_topic"})
save_df(cm_out, OUTDIR / "topic_confusion_matrix.tsv")

conf_pairs_df = make_confusion_pairs(cm_df)
conf_pairs_df.insert(0, "species", SPECIES)
save_df(conf_pairs_df, OUTDIR / "top_confusion_pairs.tsv")

plt.figure(figsize=(10, 8))
plt.imshow(cm_df.values, aspect="auto")
plt.title(f"Topic confusion matrix ({SPECIES})")
plt.xlabel("Predicted topic")
plt.ylabel("True topic")
plt.tight_layout()
plt.savefig(OUTDIR / "topic_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

display(conf_pairs_df.head(20))

saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/topic_confusion_matrix.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/top_confusion_pairs.tsv


,species,true_topic,most_confused_with,count
0,human,Topic7,Topic90,5653
1,human,Topic65,Topic90,2792
2,human,Topic31,Topic90,2055
3,human,Topic14,Topic90,877
4,human,Topic100,Topic90,784
5,human,Topic17,Topic90,657
6,human,Topic38,Topic90,585
7,human,Topic10,Topic90,583
8,human,Topic1,Topic90,579
9,human,Topic30,Topic90,526


## 10. Improved cell-type scoring

In [12]:
CELLTYPE_SCORE_STRATEGIES = [
    "mean_all",
    "mean_usable",
    "max_topic",
    "top2_mean",
    "weighted_stage0",
]

all_strategy_summaries = []
best_tables = {}

for strategy in CELLTYPE_SCORE_STRATEGIES:
    print("\n=== strategy:", strategy, "===")

    cellscore_df = compute_celltype_scores(
        eval_df=eval_df,
        topic_anno=topic_anno,
        class_names=class_names,
        strategy=strategy,
    )

    summary, scored = summarize_celltype_scores(
        cellscore_df=cellscore_df,
        topic_anno=topic_anno,
        species=SPECIES,
        strategy=strategy,
    )

    all_strategy_summaries.append(summary)
    best_tables[strategy] = scored

    save_df(summary, OUTDIR / f"celltype_projection_summary__{strategy}.tsv")

    keep_cols = [
        "best_celltype",
        "best_celltype_score",
        "second_best_celltype",
        "second_best_celltype_score",
        "celltype_margin_top1_top2",
    ]
    region_strategy = pd.concat(
        [
            eval_df[["region", "true_topic", "top_topic_pred", "top_score_pred", "margin_top1_top2_pred"]],
            scored[keep_cols],
        ],
        axis=1,
    )
    save_df(region_strategy.reset_index(drop=True), OUTDIR / f"region_level_celltype_projection__{strategy}.tsv")

celltype_projection_summary_all_strategies = pd.concat(all_strategy_summaries, axis=0, ignore_index=True)
save_df(celltype_projection_summary_all_strategies, OUTDIR / "celltype_projection_summary_all_strategies.tsv")

display(celltype_projection_summary_all_strategies.head(60))


=== strategy: mean_all ===
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/celltype_projection_summary__mean_all.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/region_level_celltype_projection__mean_all.tsv

=== strategy: mean_usable ===
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/celltype_projection_summary__mean_usable.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/region_level_celltype_projection__mean_usable.tsv

=== strategy: max_topic ===
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/celltype_projection_summary__max_topic.tsv
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/region_level_celltype_projection__max_topic.tsv

=== strategy: top2_mean ===
saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xin

,species,strategy,celltype,n_topics_assigned_in_stage0,n_usable_topics,mean_projected_score,median_projected_score,n_regions_best,mean_best_margin_when_winning
0,human,mean_all,MGE_interneuron,2,1,0.116703,0.113452,25573,0.057240
1,human,mean_all,LGE_FOXP2_TSHZ1_MSN,7,0,0.085399,0.084245,11096,0.029360
2,human,mean_all,OPC,9,1,0.023722,0.014980,1136,0.042634
3,human,mean_all,Microglia,3,1,0.024967,0.018361,640,0.042578
4,human,mean_all,Vascular,3,1,0.039782,0.034227,227,0.022135
5,human,mean_all,CGE_interneuron,6,1,0.029997,0.026296,32,0.013240
6,human,mean_all,dorsal_RG,3,3,0.021487,0.017600,22,0.020942
7,human,mean_all,TriIPC_Astrocyte,2,1,0.040179,0.036012,14,0.004623
8,human,mean_all,LGE_FOXP1_ISL1_MSN,3,0,0.015602,0.013093,6,0.012802
9,human,mean_all,ExNeu_Non_IT,4,3,0.011829,0.009028,4,0.011736


## 11. Compare scoring strategies

In [13]:
strategy_compare = (
    celltype_projection_summary_all_strategies
    .pivot_table(
        index="celltype",
        columns="strategy",
        values="n_regions_best",
        fill_value=0
    )
    .reset_index()
)

save_df(strategy_compare, OUTDIR / "celltype_strategy_comparison_n_regions_best.tsv")
display(strategy_compare)

saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/celltype_strategy_comparison_n_regions_best.tsv


strategy,celltype,max_topic,mean_all,mean_usable,top2_mean,weighted_stage0
0,CGE_LGE_progenitors,22.0,0.0,53.0,6.0,0.0
1,CGE_interneuron,45.0,32.0,114.0,86.0,16.0
2,ExIPC,4.0,0.0,0.0,2.0,1.0
3,ExNeu_IT,4.0,0.0,0.0,3.0,0.0
4,ExNeu_Non_IT,0.0,4.0,0.0,0.0,5.0
5,LGE_FOXP1_ISL1_MSN,0.0,6.0,2.0,0.0,1.0
6,LGE_FOXP1_PENK_MSN,2.0,0.0,0.0,3.0,0.0
7,LGE_FOXP2_TSHZ1_MSN,28556.0,11096.0,2689.0,30167.0,3892.0
8,LGE_OB_interneuron,4774.0,0.0,0.0,6561.0,0.0
9,MGE_interneuron,3812.0,25573.0,35480.0,667.0,34542.0


In [14]:
for strategy in CELLTYPE_SCORE_STRATEGIES:
    sub = celltype_projection_summary_all_strategies.query("strategy == @strategy").copy()
    sub = sub.sort_values("n_regions_best", ascending=False)

    plt.figure(figsize=(10, 5))
    plt.bar(sub["celltype"], sub["n_regions_best"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of regions assigned")
    plt.title(f"Cell-type projection: {SPECIES}, {strategy}")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"celltype_distribution__{strategy}.png", dpi=300, bbox_inches="tight")
    plt.close()

In [15]:
plot_df = strategy_compare.set_index("celltype")
plt.figure(figsize=(8, max(4, 0.35 * plot_df.shape[0])))
plt.imshow(plot_df.values, aspect="auto")
plt.yticks(range(plot_df.shape[0]), plot_df.index)
plt.xticks(range(plot_df.shape[1]), plot_df.columns, rotation=45, ha="right")
plt.colorbar(label="n regions best")
plt.title(f"Cell-type projection strategy comparison ({SPECIES})")
plt.tight_layout()
plt.savefig(OUTDIR / "celltype_strategy_comparison_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()

## 12. Stage 0 ↔ Stage 1 bridge

In [16]:
topic_stage0_stage1_bridge = per_topic.merge(
    topic_anno,
    how="left",
    on=["topic"],
    suffixes=("_stage1", "_stage0"),
)

save_df(topic_stage0_stage1_bridge, OUTDIR / "topic_stage0_stage1_bridge.tsv")
display(topic_stage0_stage1_bridge.head())

saved: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_model_summary/human/topic_stage0_stage1_bridge.tsv


,species_stage1,topic,n_regions,top1_accuracy,top3_accuracy,mean_true_topic_score,median_true_topic_score,mean_pred_margin_top1_top2,top1_celltype_stage1,lineage_group_stage1,...,specificity_ratio_top1_over_top2,score_delta_top1_minus_top2,annotation_confidence_stage0,annotation_class,lineage_group_stage0,lineage_group_top2,bed_path,bed_exists,n_peaks,usable_for_motif_stage0
0,human,Topic80,263,0.0,0.076046,0.075638,0.063478,0.187237,CGE_LGE_progenitors,progenitor,...,1.688774,0.015252,high,clean_top1,progenitor,progenitor,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic80.bed,True,10755,True
1,human,Topic9,146,0.0,0.027397,0.073415,0.061016,0.202320,ExIPC,progenitor,...,5.046736,0.057763,high,clean_top1,progenitor,progenitor,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic9.bed,True,10615,True
2,human,Topic5,135,0.0,0.022222,0.057701,0.052871,0.120620,ExNeu_IT,excitatory_neuron,...,1.721155,0.011350,high,clean_top1,excitatory_neuron,excitatory_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic5.bed,True,10672,True
3,human,Topic60,92,0.0,0.010870,0.055252,0.043518,0.041840,dorsal_RG,progenitor,...,10.253764,0.099623,high,clean_top1,progenitor,glia_like_progenitor,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic60.bed,True,14894,True
4,human,Topic95,117,0.0,0.008547,0.086701,0.085106,0.098524,TriIPC_Astrocyte,glia_like_progenitor,...,1.751865,0.029418,high,clean_top1,glia_like_progenitor,glia,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic95.bed,True,28715,True


## 13. Outputs

Key tables:

- `overall_metrics.tsv`
- `region_level_predictions.tsv`
- `true_vs_pred_topic_counts.tsv`
- `per_topic_summary.tsv`
- `stage2_topic_candidates.tsv`
- `topic_confusion_matrix.tsv`
- `top_confusion_pairs.tsv`
- `celltype_projection_summary_all_strategies.tsv`
- `celltype_strategy_comparison_n_regions_best.tsv`
- `region_level_celltype_projection__mean_all.tsv`
- `region_level_celltype_projection__mean_usable.tsv`
- `region_level_celltype_projection__max_topic.tsv`
- `region_level_celltype_projection__top2_mean.tsv`
- `region_level_celltype_projection__weighted_stage0.tsv`

Key figures:

- `topic_confusion_matrix.png`
- `celltype_distribution__mean_all.png`
- `celltype_distribution__mean_usable.png`
- `celltype_distribution__max_topic.png`
- `celltype_distribution__top2_mean.png`
- `celltype_distribution__weighted_stage0.png`
- `celltype_strategy_comparison_heatmap.png`